# 多因子分析模板

本 notebook 用于分析多个已入库因子之间的关系，
检测冗余、评估边际贡献，为因子组合和模型构建做准备。

## 工作流
1. 环境设置
2. 加载多因子（FactorStore / FactorCatalog）
3. 因子相关性矩阵
4. VIF 多重共线性检测
5. 增量 IC 分析
6. 冗余因子筛除建议
7. 因子组合初探

## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from factor_research.evaluation.correlation import correlation_analysis, incremental_ic
from factor_research.evaluation.ic import ic_series, ic_summary
from factor_research.evaluation.metrics import compute_forward_returns_panel, spearman_ic
from factor_research.store.factor_store import FactorStore
from factor_research.store.catalog import FactorCatalog

plt.rcParams["figure.figsize"] = (10, 8)
np.random.seed(42)
print("环境加载完成")

## 2. 加载多因子

### 方式 A: 合成数据（无需数据库）

In [ ]:
# === 合成多因子数据 ===
n = 2000
symbols = ["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT", "DOGE/USDT"]
index = pd.date_range("2024-01-01", periods=n, freq="1min", tz="UTC")

# 共同信号
common = np.random.randn(n)

# 价格面板
price_data = {}
for sym in symbols:
    noise = np.random.randn(n) * 0.3
    price_data[sym] = 100 * np.exp(np.cumsum((common + noise) * 0.001))
price_panel = pd.DataFrame(price_data, index=index)

# 因子面板: 5 个因子，其中 factor_1 和 factor_2 高度相关
factor_panels = {}

# factor_1: 动量信号
factor_panels["momentum_5m"] = pd.DataFrame(
    {sym: common + np.random.randn(n) * 0.3 for sym in symbols}, index=index,
)

# factor_2: 动量信号的变体（高冗余）
factor_panels["momentum_10m"] = pd.DataFrame(
    {sym: common * 0.9 + np.random.randn(n) * 0.1 for sym in symbols}, index=index,
)

# factor_3: 独立波动率因子
factor_panels["volatility"] = pd.DataFrame(
    {sym: np.random.randn(n) for sym in symbols}, index=index,
)

# factor_4: 微结构因子（与动量有弱相关）
factor_panels["microstructure"] = pd.DataFrame(
    {sym: common * 0.2 + np.random.randn(n) * 0.8 for sym in symbols}, index=index,
)

# factor_5: 纯噪声（无效因子）
factor_panels["noise"] = pd.DataFrame(
    {sym: np.random.randn(n) for sym in symbols}, index=index,
)

print(f"已加载 {len(factor_panels)} 个因子: {list(factor_panels.keys())}")
print(f"价格面板: {price_panel.shape}")

In [ ]:
# === 方式 B: 从 FactorStore 加载已入库因子 ===
# store = FactorStore()
# catalog = FactorCatalog(store=store)
#
# # 浏览目录
# print(catalog.summary())
#
# # 加载因子
# factor_panels = {}
# for name in ["returns_5m", "returns_10m", "orderbook_imbalance"]:
#     if name in catalog:
#         factor_panels[name] = store.load(name)
#
# # 从 DataReader 加载价格
# from data_infra.data.reader import DataReader
# reader = DataReader()
# price_data = {}
# for sym in symbols:
#     ohlcv = reader.get_ohlcv(sym, timeframe="1m", limit=2000)
#     price_data[sym] = ohlcv.set_index("timestamp")["close"]
# price_panel = pd.DataFrame(price_data)
print("如需从 FactorStore 加载，请取消上方注释")

### 方式 C: 按因子族浏览（族级检索）

使用 `FactorCatalog` 的族功能，快速查看哪些因子属于同一参数族，比较不同参数变体。

In [ ]:
# === 族级检索示例 ===
# store = FactorStore()
# catalog = FactorCatalog(store=store)
#
# # 查看所有已入库的因子族
# print("已注册因子族:", catalog.list_families())
#
# # 查看某个族的参数变体对比
# summary = catalog.family_summary("multi_scale_returns")
# print(summary)  # 每行一个变体，列含参数值 + 存储状态
#
# # 按族名筛选因子
# momentum_factors = catalog.search(family="multi_scale_returns")
# print(momentum_factors)
#
# # 批量加载整个族的因子面板
# family_panels = store.load_family("multi_scale_returns")
# for name, panel in family_panels.items():
#     factor_panels[name] = panel
#     print(f"已加载 {name}: {panel.shape}")
print("族级检索示例代码已注释，取消注释后执行")

## 3. 因子相关性矩阵

In [ ]:
corr_result = correlation_analysis(factor_panels)
corr_matrix = corr_result["correlation_matrix"]

print("因子相关性矩阵 (Spearman):")
print(corr_matrix.round(3))

In [ ]:
# 热力图
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, ax=ax,
)
ax.set_title("Factor Correlation Matrix (Spearman)")
plt.tight_layout()
plt.show()

## 4. VIF 多重共线性检测

VIF 经验阈值:
- VIF < 5: 可接受
- 5 ≤ VIF < 10: 有共线性嫌疑
- VIF ≥ 10: 严重共线性，建议剔除

In [ ]:
vif = corr_result["vif"]
print("VIF (方差膨胀因子):")
for name, val in sorted(vif.items(), key=lambda x: -x[1]):
    flag = " ⚠️" if val >= 5 else ""
    print(f"  {name:20s}: {val:.2f}{flag}")

## 5. 增量 IC 分析

评估每个因子相对于其他因子的边际贡献。
如果增量 IC 远小于原始 IC，说明该因子的信息已被其他因子覆盖。

In [ ]:
# 逐个因子计算增量 IC
incr_results = {}
factor_names = list(factor_panels.keys())

for name in factor_names:
    new = factor_panels[name]
    existing = {k: v for k, v in factor_panels.items() if k != name}
    result = incremental_ic(new, existing, price_panel, horizon=1)
    incr_results[name] = result

# 汇总表
incr_df = pd.DataFrame(incr_results).T
print("增量 IC 分析:")
print(incr_df.round(4))

## 6. 冗余因子筛除建议

In [ ]:
print("=== 冗余因子筛除建议 ===")
print()

# 规则 1: VIF > 5 的因子
high_vif = [name for name, val in vif.items() if val >= 5]
if high_vif:
    print(f"高 VIF 因子 (≥5): {high_vif}")
    print("  → 建议保留组内 IC 最高的因子，剔除其余\n")

# 规则 2: 高相关因子对 (|corr| > 0.7)
print("高相关因子对 (|corr| > 0.7):")
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            n1, n2 = corr_matrix.index[i], corr_matrix.columns[j]
            print(f"  {n1} ↔ {n2}: {corr_matrix.iloc[i, j]:.3f}")

# 规则 3: 信息保留率 < 0.3 的因子
print("\n低增量 IC 因子 (info_retention < 0.3):")
for name, res in incr_results.items():
    ret = res.get("info_retention", np.nan)
    if not np.isnan(ret) and ret < 0.3:
        print(f"  {name}: info_retention = {ret:.3f}")

## 7. 因子组合初探

简单等权组合 vs 单因子的 IC 比较。
等权组合通常能提升稳定性（降低 IC 波动），即使 IC 均值不变。

In [ ]:
# 等权组合: 将所有因子标准化后等权平均
standardized = {}
for name, panel in factor_panels.items():
    # 截面标准化
    mean = panel.mean(axis=1)
    std = panel.std(axis=1).replace(0, 1)
    standardized[name] = panel.sub(mean, axis=0).div(std, axis=0)

# 等权平均
combined = sum(standardized.values()) / len(standardized)

# 计算组合因子的 IC
combo_ic_ts = ic_series(combined, price_panel, horizon=1)
combo_summary = ic_summary(combo_ic_ts)

# 对比
print("=== 单因子 vs 等权组合 IC ===")
print(f"{'因子':20s} {'IC_mean':>10s} {'IC_IR':>10s}")
print("-" * 42)
for name in factor_names:
    single_ic = ic_series(factor_panels[name], price_panel, horizon=1)
    single_summary = ic_summary(single_ic)
    print(f"{name:20s} {single_summary['ic_mean']:>+10.4f} {single_summary['ic_ir']:>+10.4f}")
print("-" * 42)
print(f"{'等权组合':20s} {combo_summary['ic_mean']:>+10.4f} {combo_summary['ic_ir']:>+10.4f}")